In [2]:
import logging
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, days
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType

In [3]:
def create_spark_session(app_name: str) -> SparkSession:
    spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName(app_name)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.lakehouse_prod","org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lakehouse_prod.type", "hive")
        .config("spark.sql.catalog.lakehouse_prod.uri","thrift://hive-metastore:9083")
        .config("spark.sql.catalog.lakehouse_prod.warehouse","s3a://lakehouse-prod-bucket/warehouse/")
        .config("spark.sql.catalog.lakehouse_prod.io-impl","org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .enableHiveSupport()
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")
    print("NOTE: SparkSession created successfully!")

    return spark

In [4]:
app_name = 'Spark-Iceberg-Test'
spark = create_spark_session(app_name)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 16:34:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


NOTE: SparkSession created successfully!


In [4]:
sc = spark.sparkContext
sc

<SparkContext master=spark://spark-master:7077 appName=Spark-Iceberg-Test>

In [6]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse_prod.db").show()

++
||
++
++



In [5]:
# Initial Database Creation
try:
    spark = create_spark_session("Create Initial Database")

    # Create databases for bronze, silver, and gold layers in the lakehouse
    spark.sql("""
            CREATE DATABASE IF NOT EXISTS lakehouse_prod.bronze_db
            LOCATION 's3a://lakehouse-gold-bucket/warehouse/bronze'
    """)

    spark.sql("""
            CREATE DATABASE IF NOT EXISTS lakehouse_prod.silver_db
            LOCATION 's3a://lakehouse-gold-bucket/warehouse/silver'
    """)

    spark.sql("""
            CREATE DATABASE IF NOT EXISTS lakehouse_prod.gold_db
            LOCATION 's3a://lakehouse-gold-bucket/warehouse/gold'
    """)

except Exception as e:
    print(f"Job failed: {e}")

finally:
    if spark:
        spark.stop()

NOTE: SparkSession created successfully!


In [6]:
spark.sql("SHOW DATABASES IN lakehouse_prod").show()

+---------+
|namespace|
+---------+
|bronze_db|
|       db|
|  default|
|  gold_db|
|      nyc|
|silver_db|
+---------+



In [13]:
spark.sql("USE lakehouse_prod").show()

++
||
++
++



In [ ]:
spark.sql("SHOW CATALOGS;").show()

In [ ]:
spark.sql("SHOW NAMESPACES").show()

In [ ]:
spark.sql("SHOW TABLES in nyc;").show()

In [ ]:
# spark.sql("""
# DROP DATABASE nyc
# """).show()

In [7]:
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse_prod.db").show()

++
||
++
++



In [10]:
spark.sql("SHOW DATABASES;").show()

+---------+
|namespace|
+---------+
|       db|
|  default|
|      nyc|
+---------+



In [ ]:
spark.sql(
"""
CREATE TABLE IF NOT EXISTS lakehouse_prod.nyc.taxis
(
  trip_id bigint,
  trip_distance float,
  fare_amount double,
  store_and_fwd_flag string
)
USING iceberg
LOCATION 's3a://lakehouse-dev-bucket/test_iceberg/nyc_taxis';

"""
).show()

In [14]:
spark.sql("""
DESCRIBE EXTENDED lakehouse_prod.nyc.taxis_02
""").show(truncate=False)

+---------------------+-------------+-------+
|col_name             |data_type    |comment|
+---------------------+-------------+-------+
|VendorID             |bigint       |NULL   |
|tpep_pickup_datetime |timestamp_ntz|NULL   |
|tpep_dropoff_datetime|timestamp_ntz|NULL   |
|passenger_count      |double       |NULL   |
|trip_distance        |double       |NULL   |
|RatecodeID           |double       |NULL   |
|store_and_fwd_flag   |string       |NULL   |
|PULocationID         |bigint       |NULL   |
|DOLocationID         |bigint       |NULL   |
|payment_type         |bigint       |NULL   |
|fare_amount          |double       |NULL   |
|extra                |double       |NULL   |
|mta_tax              |double       |NULL   |
|tip_amount           |double       |NULL   |
|tolls_amount         |double       |NULL   |
|improvement_surcharge|double       |NULL   |
|total_amount         |double       |NULL   |
|congestion_surcharge |double       |NULL   |
|airport_fee          |double     

In [15]:
spark.sql("SHOW CATALOGS").show()

+--------------+
|       catalog|
+--------------+
|lakehouse_prod|
| spark_catalog|
+--------------+



In [16]:
# Define schema for NYC Taxi Data
schema = StructType([
    StructField('VendorID', LongType(), True),
    StructField('tpep_pickup_datetime', TimestampNTZType(), True),
    StructField('tpep_dropoff_datetime', TimestampNTZType(), True),
    StructField('passenger_count', DoubleType(), True),
    StructField('trip_distance', DoubleType(), True),
    StructField('RatecodeID', DoubleType(), True),
    StructField('store_and_fwd_flag', StringType(), True),
    StructField('PULocationID', LongType(), True),
    StructField('DOLocationID', LongType(), True),
    StructField('payment_type', LongType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('extra', DoubleType(), True),
    StructField('mta_tax', DoubleType(), True),
    StructField('tip_amount', DoubleType(), True),
    StructField('tolls_amount', DoubleType(), True),
    StructField('improvement_surcharge', DoubleType(), True),
    StructField('total_amount', DoubleType(), True),
    StructField('congestion_surcharge', DoubleType(), True),
    StructField('airport_fee', DoubleType(), True)
])

In [17]:
# Read Parquet file from MinIO
parquet_path = './data/yellow_tripdata_2022-04.parquet'
df = spark.read.option("header", "true").schema(schema).parquet(parquet_path)
df.show(10)

[Stage 2:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [ ]:
# df.write.mode("overwrite").saveAsTable("nyc.yellow_trip_data_202204")

In [6]:
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [ ]:
# df.writeTo("lakehouse_prod.nyc.taxis_01") \
#   .option("location", "s3a://lakehouse-dev-bucket/external_test/taxis_01") \
#   .partitionedBy(days("tpep_pickup_datetime")) \
#   .replace()

In [ ]:
df.writeTo("lakehouse_prod.nyc.taxis_01") \
  .option("location", "s3a://lakehouse-dev-bucket/external_test/taxis_01") \
  .partitionedBy(days("tpep_pickup_datetime")) \
  .create()

In [5]:
spark.sql("""
    SELECT * FROM lakehouse_prod.nyc.taxis_01
""").show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [7]:
spark.sql("""
    SHOw TABLES in nyc;
""").toPandas()

,namespace,tableName,isTemporary
0,nyc,taxis,False
1,nyc,yellow_trip_data_202204,False
2,nyc,taxis_01,False
3,nyc,taxis_02,False
4,nyc,breweries,False


In [ ]:
spark.sql("""
    SELECT * FROM lakehouse_prod.nyc.taxis_01
""").show(5)

In [10]:
spark.sql("""
    SELECT * FROM nyc.yellow_trip_data_202204
""").show(5)

[Stage 1:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [9]:
df.writeTo("lakehouse_prod.nyc.taxis_02") \
    .tableProperty("location", "s3a://lakehouse-dev-bucket/external_test/taxis_02") \
    .createOrReplace()

In [10]:
df.writeTo("lakehouse_prod.nyc.taxis_02") \
    .overwritePartitions()

In [11]:
df = spark.read.format("iceberg").load("lakehouse_prod.nyc.taxis_02")
df.show()

[Stage 4:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [28]:
df.take(5)

[Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2022, 4, 1, 0, 21, 13), tpep_dropoff_datetime=datetime.datetime(2022, 4, 1, 0, 58, 33), passenger_count=1.0, trip_distance=10.3, RatecodeID=1.0, store_and_fwd_flag='N', PULocationID=163, DOLocationID=62, payment_type=1, fare_amount=33.5, extra=3.0, mta_tax=0.5, tip_amount=7.45, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=44.75, congestion_surcharge=2.5, airport_fee=0.0),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2022, 4, 1, 0, 7, 47), tpep_dropoff_datetime=datetime.datetime(2022, 4, 1, 0, 19, 12), passenger_count=0.0, trip_distance=2.0, RatecodeID=1.0, store_and_fwd_flag='N', PULocationID=142, DOLocationID=141, payment_type=1, fare_amount=10.0, extra=3.0, mta_tax=0.5, tip_amount=4.1, tolls_amount=0.0, improvement_surcharge=0.3, total_amount=17.9, congestion_surcharge=2.5, airport_fee=0.0),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2022, 4, 1, 0, 14, 52), tpep_dropoff_datetime=datetime.date

In [29]:
df.limit(5).show()

[Stage 24:>                                                         (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [35]:
df_50 = df.limit(5)
df_50.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [36]:
# Test for Delete
df_50.writeTo("lakehouse_prod.nyc.taxitables_test_delete") \
    .tableProperty("location", "s3a://lakehouse-dev-bucket/external_test/taxitables_test_delete") \
    .partitionedBy(days("tpep_pickup_datetime")) \
    .createOrReplace()

In [41]:
spark.sql("""
    SELECT * FROM lakehouse_prod.nyc.taxitables_test_delete
""").show(5)

++
||
++
++



In [42]:
spark.sql("""
    DROP TABLE lakehouse_prod.nyc.taxitables_test_delete;
""")

DataFrame[]

In [39]:
spark.sql("""
CREATE TABLE lakehouse_prod.nyc.taxitables_test_delete
USING ICEBERG
LOCATION 's3a://lakehouse-dev-bucket/external_test/taxitables_test_delete';
""")

DataFrame[]

In [24]:
spark.sql("""
    DROP TABLE lakehouse_prod.nyc.taxitables_test_delete PURGE;
""")

++
||
++
++



In [20]:
DROP TABLE lakehouse.db.sales PURGE;

SyntaxError: invalid syntax (494671604.py, line 1)

In [ ]:
spark.stop()